In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

bc = pd.read_csv(ROOT / "data" / "staging" / "budget_categories.csv")
bc["t"] = bc["year"] * 12 + bc["month"]

periods = sorted(bc["t"].unique())
hh = sorted(bc["user_profile_id"].unique())
cats = sorted(bc["category"].unique())
idx = pd.MultiIndex.from_product([hh, periods], names=["user_profile_id", "t"])

def cube(col):
    w = bc.pivot_table(index=["user_profile_id", "t"], columns="category", values=col, aggfunc="sum")
    w = w.reindex(index=idx, columns=cats).fillna(0)
    return w.to_numpy().reshape(len(hh), len(periods), len(cats))

A, B = cube("actual_kes"), cube("allocated_kes")
print("A and B shape (households, months, categories):", A.shape)

A and B shape (households, months, categories): (500, 18, 17)


In [7]:
TEST = range(12, 18)  # month positions 13-18, each forecast from the months before it

def score(forecast_fn, name):
    ae_t, ae_c, act = [], [], []
    for t in TEST:
        f, a = forecast_fn(t), A[:, t, :]
        ae_t.append(np.abs(f.sum(axis=1) - a.sum(axis=1)))
        ae_c.append(np.abs(f - a).sum(axis=1))
        act.append(a.sum(axis=1))
    ae_t, ae_c, act = map(np.concatenate, (ae_t, ae_c, act))
    return {"method": name, "MAE_KES": round(ae_t.mean()),
            "WAPE": round(ae_t.sum() / act.sum(), 3),
            "category_WAPE": round(ae_c.sum() / act.sum(), 3)}

results = [
    score(lambda t: np.median(A[:, t - 12:t, :], axis=1), "median_12m"),
    score(lambda t: 0.75 * B[:, t, :] + 0.25 * np.median(A[:, t - 12:t, :], axis=1), "blend_75_25"),
]
pd.DataFrame(results)

,method,MAE_KES,WAPE,category_WAPE
0,median_12m,9180,0.123,0.255
1,blend_75_25,7635,0.103,0.222


In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

H, T, C = A.shape
cat_id = np.broadcast_to(np.arange(C), (H, C))

def features(t):
    """One row per (household, category) for month t, using only months before t plus t's budget."""
    hist = A[:, max(0, t - 12):t, :]
    cols = [
        B[:, t, :],                                       # this month's budget
        A[:, t - 1, :], A[:, t - 2, :], A[:, t - 3, :],   # the last three months
        np.median(hist, axis=1),                          # usual level (median, up to 12 months)
        hist.mean(axis=1),                                # average level
        B[:, t - 1, :],                                   # last month's budget
        cat_id,                                           # which category
    ]
    return np.stack([c.reshape(-1) for c in cols], axis=1)

def gbm_forecast(t):
    # Train only on months before t, so the model never sees the month it is forecasting.
    X_train = np.vstack([features(s) for s in range(3, t)])
    y_train = np.concatenate([A[:, s, :].reshape(-1) for s in range(3, t)])
    model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05,
                                          categorical_features=[7], random_state=0)
    model.fit(X_train, y_train)
    return np.clip(model.predict(features(t)), 0, None).reshape(H, C)

results.append(score(gbm_forecast, "boosted_tree"))
pd.DataFrame(results)

,method,MAE_KES,WAPE,category_WAPE
0,median_12m,9180,0.123,0.255
1,blend_75_25,7635,0.103,0.222
2,boosted_tree,8567,0.115,0.208


In [9]:
def gbm_forecast_mean(t):
    X_train = np.vstack([features(s) for s in range(3, t)])
    y_train = np.concatenate([A[:, s, :].reshape(-1) for s in range(3, t)])
    model = HistGradientBoostingRegressor(loss="squared_error", max_iter=300, learning_rate=0.05,
                                          categorical_features=[7], random_state=0)
    model.fit(X_train, y_train)
    return np.clip(model.predict(features(t)), 0, None).reshape(H, C)

results.append(score(gbm_forecast_mean, "boosted_tree_mean"))
pd.DataFrame(results)

,method,MAE_KES,WAPE,category_WAPE
0,median_12m,9180,0.123,0.255
1,blend_75_25,7635,0.103,0.222
2,boosted_tree,8567,0.115,0.208
3,boosted_tree_mean,8155,0.110,0.237


In [10]:
import torch
import torch.nn as nn

L = 6  # months of history the LSTM reads, in order

class Forecaster(nn.Module):
    def __init__(self, n_cat, hidden=64):
        super().__init__()
        # Reads a sequence of months; each month = actual spend + budget for every category.
        self.lstm = nn.LSTM(input_size=2 * n_cat, hidden_size=hidden, batch_first=True)
        # Combines the LSTM's memory with this month's budget to predict each category.
        self.head = nn.Sequential(nn.Linear(hidden + n_cat, 64), nn.ReLU(), nn.Linear(64, n_cat))

    def forward(self, seq, next_budget):
        _, (h, _) = self.lstm(seq)
        return self.head(torch.cat([h[-1], next_budget], dim=1))

def lstm_forecast(t, epochs=300):
    torch.manual_seed(0)
    scale = A[:, :t, :].mean(axis=(0, 1)) + 1.0   # per-category size, from training months only

    def build(s):
        seq = np.concatenate([A[:, s - L:s, :], B[:, s - L:s, :]], axis=2) / np.concatenate([scale, scale])
        return seq, B[:, s, :] / scale, A[:, s, :] / scale

    parts = [build(s) for s in range(L, t)]       # training examples: months before t only
    seq = torch.tensor(np.concatenate([p[0] for p in parts]), dtype=torch.float32)
    nb = torch.tensor(np.concatenate([p[1] for p in parts]), dtype=torch.float32)
    y = torch.tensor(np.concatenate([p[2] for p in parts]), dtype=torch.float32)

    model = Forecaster(C)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(seq, nb), y)
        loss.backward()
        opt.step()

    test_seq, test_nb, _ = build(t)
    with torch.no_grad():
        pred = model(torch.tensor(test_seq, dtype=torch.float32),
                     torch.tensor(test_nb, dtype=torch.float32)).numpy()
    return np.clip(pred * scale, 0, None)

results.append(score(lstm_forecast, "lstm"))
pd.DataFrame(results)

,method,MAE_KES,WAPE,category_WAPE
0,median_12m,9180,0.123,0.255
1,blend_75_25,7635,0.103,0.222
2,boosted_tree,8567,0.115,0.208
3,boosted_tree_mean,8155,0.110,0.237
4,lstm,9974,0.134,0.253


In [11]:
# Remember each model's forecast so nothing is retrained twice.
cache = {}
def cached(fn, name):
    def f(t):
        if (name, t) not in cache:
            cache[(name, t)] = fn(t)
        return cache[(name, t)]
    return f

blend = cached(lambda t: 0.75 * B[:, t, :] + 0.25 * np.median(A[:, t - 12:t, :], axis=1), "blend")
tree = cached(gbm_forecast, "tree")
lstm = cached(lstm_forecast, "lstm")

def hybrid(t):
    """Total from the blend, split across categories the way the boosted tree predicts."""
    b, g = blend(t), tree(t)
    share = g / np.maximum(g.sum(axis=1, keepdims=True), 1e-9)
    return share * b.sum(axis=1, keepdims=True)

candidates = {
    "blend_75_25": blend,
    "boosted_tree": tree,
    "lstm": lstm,
    "hybrid: blend total, tree split": hybrid,
    "average of blend + tree": lambda t: (blend(t) + tree(t)) / 2,
    "average of blend + tree + lstm": lambda t: (blend(t) + tree(t) + lstm(t)) / 3,
}

def score_months(fn, name, months):
    ae_t, ae_c, act = [], [], []
    for t in months:
        f, a = fn(t), A[:, t, :]
        ae_t.append(np.abs(f.sum(axis=1) - a.sum(axis=1)))
        ae_c.append(np.abs(f - a).sum(axis=1))
        act.append(a.sum(axis=1))
    ae_t, ae_c, act = map(np.concatenate, (ae_t, ae_c, act))
    return {"method": name, "MAE_KES": round(ae_t.mean()),
            "WAPE": round(ae_t.sum() / act.sum(), 3),
            "category_WAPE": round(ae_c.sum() / act.sum(), 3)}

validation = pd.DataFrame([score_months(fn, n, range(12, 15)) for n, fn in candidates.items()])
final_test = pd.DataFrame([score_months(fn, n, range(15, 18)) for n, fn in candidates.items()])
print("CHOOSE on months 13-15:"); display(validation)
print("CONFIRM on months 16-18:"); display(final_test)

CHOOSE on months 13-15:


,method,MAE_KES,WAPE,category_WAPE
0,blend_75_25,7340,0.099,0.218
1,boosted_tree,8218,0.111,0.204
2,lstm,9363,0.126,0.247
3,"hybrid: blend total, tree split",7340,0.099,0.213
4,average of blend + tree,7399,0.100,0.208
5,average of blend + tree + lstm,7450,0.101,0.216


CONFIRM on months 16-18:


,method,MAE_KES,WAPE,category_WAPE
0,blend_75_25,7931,0.106,0.226
1,boosted_tree,8917,0.119,0.212
2,lstm,10586,0.142,0.258
3,"hybrid: blend total, tree split",7931,0.106,0.221
4,average of blend + tree,8074,0.108,0.217
5,average of blend + tree + lstm,8262,0.110,0.225
